**Install Required Libraries**

In [50]:
!pip install groq sentence-transformers faiss-cpu pypdf numpy matplotlib scikit-learn -q

**Importing Libraries**

In [51]:
import os
import numpy as np
import faiss

from groq import Groq
from sentence_transformers import SentenceTransformer
from pypdf import PdfReader

**Setup Groq API**

In [53]:
from getpass import getpass
GROQ_API_KEY = getpass("Enter your GROQ API Key: ")

Enter your GROQ API Key: ··········


In [54]:
client = Groq(api_key = GROQ_API_KEY)
MODEL = "openai/gpt-oss-120b"

**Ask LLM Without RAG**

In [55]:
messages = [{"role": "user", "content": "What is inside my private PDF?"}]
response = client.chat.completions.create(model=MODEL, messages=messages)
print(response)
print(response.choices)
print(response.choices[0])
print(response.choices[0].message)
print("+"*50)
print(response.choices[0].message.content)

ChatCompletion(id='chatcmpl-7d39b849-88d5-45e5-83f8-1e9b4d638be5', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='I’m not able to see the contents of a PDF unless you share the file with me here. If you’d like me to look at it, please upload the PDF (or the relevant pages) and let me know what you’d like to learn or extract from it. I’ll treat the information confidentially and only use it to answer your request.', role='assistant', annotations=None, executed_tools=None, function_call=None, reasoning='The user asks "What is inside my private PDF?" They likely want the assistant to read a PDF they have. There\'s no PDF provided. According to policy, we cannot access external files unless provided. The user hasn\'t provided the PDF. We can ask them to upload the PDF. Also note privacy: we should ensure confidentiality. So respond asking to upload the PDF file.', tool_calls=None))], created=1794123389, model='openai/gpt-oss-120b', obje

**Load Embedding Model**

In [56]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

**Create Sample Document**

In [57]:
documents = [
    "RAG is Retrieval Augmented Generation.",
    "Embeddings convert text into vectors.",
    "FAISS is used for similarity search.",
    "Chunking splits large documents into small pieces."
]
print(documents)

['RAG is Retrieval Augmented Generation.', 'Embeddings convert text into vectors.', 'FAISS is used for similarity search.', 'Chunking splits large documents into small pieces.']


**Generate Embeddings**

In [58]:
embeddings = embedding_model.encode(documents)
print(embeddings)

[[-0.07537757  0.03734026  0.05083374 ...  0.01072464 -0.00982657
   0.01364366]
 [ 0.00084422  0.01033457 -0.02023601 ...  0.06018496  0.09718707
  -0.04037093]
 [-0.06509122 -0.02452535 -0.07881898 ...  0.03978903  0.10179748
   0.03824164]
 [-0.05169759  0.12222588  0.0370485  ... -0.01414929  0.02006992
   0.02113875]]


In [59]:
print("Rows: ", embeddings.shape[0])
print("Columns: ", embeddings.shape[1])

Rows:  4
Columns:  384


**Creating FAISS (Facebook AI Similarity Search) Index**:
- for storing, providing index to the embedding and making the search easy and fast
- group similar vectors into clusters
- At query time, only search nearby clusters.
- Approximate, but very fast. (often, ~ O(√n)) => time complexity

*   **Name-Approach**

In [60]:
columns = embeddings.shape[1]   # dimensions=Columns = Total No. of Embeddings
# Performing Nearest Neighbour Search
index = faiss.IndexFlatL2(columns)  # Uses Euclidean distance to compare vectors, using Brute-Force Method
index.add(np.array(embeddings, dtype = "float32"))
print(embeddings)
print("Total No. of Vectors Stored inside the Search Index: ", index.ntotal)

[[-0.07537757  0.03734026  0.05083374 ...  0.01072464 -0.00982657
   0.01364366]
 [ 0.00084422  0.01033457 -0.02023601 ...  0.06018496  0.09718707
  -0.04037093]
 [-0.06509122 -0.02452535 -0.07881898 ...  0.03978903  0.10179748
   0.03824164]
 [-0.05169759  0.12222588  0.0370485  ... -0.01414929  0.02006992
   0.02113875]]
Total No. of Vectors Stored inside the Search Index:  4


* **Grouping Approach**

In [61]:
dimensions = columns = embeddings.shape[1]
nlist = 2       # 2 points in 2 clusters
quantizer = faiss.IndexFlatL2(dimensions)
index = faiss.IndexIVFFlat(   # ANN search index ho used in database systems to quickly find similar items (like images, text, or songs) out of millions of options.
    # Quantizer le fast search huna help garxa, jun cluster ho teii ma matra herxa,
    # no Time Pass, looking out for other clusters
    quantizer,  # query vector ko lagi nearest cluster vector kun ho vanera batauni kaam garxa
    dimensions, # number of columns during embedding
    nlist, # number of clusters, kati wota points laii kati wota cluster diney vanera batauxa
    faiss.METRIC_L2   # Uses Squared Euclidean distance to compare vectors, using Brute-Force Method
    )
# Training the Embeddings for Creating Clusters
index.train(embeddings)
index.add(embeddings)

**Ask a Question**

In [62]:
query = "What does RAG stands for?"
query_embedding = embedding_model.encode([query])
# query_embedding

**Search Similar Chunks**

In [63]:
distances, indices = index.search(np.array(query_embedding, dtype = "float32"), 2)
print(distances)
print(indices)

[[0.66851383 1.6689006 ]]
[[0 2]]


**Retrieve the Matching Documents**

In [64]:
retrieved_docs = []

for i in indices[0]:
  print(documents[i])
  retrieved_docs.append(documents[i])

# Using List Comprehension:
# retrieved_docs = [documents[i] for i in indices[0]]

RAG is Retrieval Augmented Generation.
FAISS is used for similarity search.


In [65]:
print(retrieved_docs)

['RAG is Retrieval Augmented Generation.', 'FAISS is used for similarity search.']


**Create Context**

In [66]:
context = "\n".join(retrieved_docs)
print(context)

RAG is Retrieval Augmented Generation.
FAISS is used for similarity search.


**Send Context to LLM**

In [67]:
messages = [
    {"role": "system", "content": "Answer only using the provided context."},
    {"role": "user", "content": f"Context: \n{context}\n\nQuestion: {query}"}
]

response = client.chat.completions.create(model = MODEL, messages = messages)
print(response)
print(response.choices)
print(response.choices[0])
print(response.choices[0].message)
print(response.choices[0].message.content)

ChatCompletion(id='chatcmpl-1bc7a6eb-127d-46a5-be2f-753027158bf1', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='RAG stands for **Retrieval Augmented Generation**.', role='assistant', annotations=None, executed_tools=None, function_call=None, reasoning='We need to answer only using provided context. The context says: "RAG is Retrieval Augmented Generation." So answer: Retrieval Augmented Generation.', tool_calls=None))], created=1794123394, model='openai/gpt-oss-120b', object='chat.completion', mcp_list_tools=None, service_tier='on_demand', system_fingerprint='fp_ca5edfaab2', usage=CompletionUsage(completion_tokens=52, prompt_tokens=109, total_tokens=161, completion_time=0.107511384, completion_tokens_details=CompletionTokensDetails(reasoning_tokens=31), prompt_time=0.004931733, prompt_tokens_details=None, queue_time=0.076566319, total_time=0.112443117), usage_breakdown=None, x_groq=XGroq(id='req_01m6x1cv2pectafbt2ph3r43m3', debug=

**RAG based on PDF Data**

**Loading PDF File**

In [68]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [69]:
filepath = r'/content/drive/MyDrive/HPN Unit Three.pdf'

In [70]:
pdf_data = PdfReader(filepath)
text = ""
for page in pdf_data.pages:
  text += page.extract_text()
print(text)

Unit -3: Performance Metrics and Network Optimization 
 
1. Measuring Network Performance 
a. Latency 
 Definition: Time taken for a data packet to travel from source to destination. 
 Types: 
o Propagation Delay: Time for signal to travel through the medium. 
o Transmission Delay: Time to push bits onto the wire. 
o Processing Delay: Time to process packet headers. 
o Queuing Delay: Time a packet waits in queue before transmission. 
b. Bandwidth 
 Definition: Maximum rate at which data can be transferred over a network path. 
 Units: Bits per second (bps), Mbps, Gbps. 
 Theoretical vs. Effective Bandwidth: 
o Theoretical: Maximum possible. 
o Effective: Real-world bandwidth after overheads. 
c. Throughput 
 Definition: Actual amount of data successfully transmitted over a network per unit time. 
 Affected by: Protocol overheads, congestion, retransmissions. 
Practical Example: 
If a 10 MB file takes 10 seconds to transfer: 
Throughput=10 MB10 sec=1 MBps=8 Mbps\text{Throughput} 

**Chunking out the PDF Text**

In [71]:
chunk_size = 300
chunks = []
for i in range(0, len(text), chunk_size):
  chunk = text[i:i + chunk_size]
  chunks.append(chunk)
  print(f"Chunk {i}: \n{chunk}")

Chunk 0: 
Unit -3: Performance Metrics and Network Optimization 
 
1. Measuring Network Performance 
a. Latency 
 Definition: Time taken for a data packet to travel from source to destination. 
 Types: 
o Propagation Delay: Time for signal to travel through the medium. 
o Transmission Delay: Time to push bi
Chunk 300: 
ts onto the wire. 
o Processing Delay: Time to process packet headers. 
o Queuing Delay: Time a packet waits in queue before transmission. 
b. Bandwidth 
 Definition: Maximum rate at which data can be transferred over a network path. 
 Units: Bits per second (bps), Mbps, Gbps. 
 Theoretical vs. E
Chunk 600: 
ffective Bandwidth: 
o Theoretical: Maximum possible. 
o Effective: Real-world bandwidth after overheads. 
c. Throughput 
 Definition: Actual amount of data successfully transmitted over a network per unit time. 
 Affected by: Protocol overheads, congestion, retransmissions. 
Practical Example: 
I
Chunk 900: 
f a 10 MB file takes 10 seconds to transfer: 
Throu

In [72]:
 print("Total Chunks:\n", len(chunks))

Total Chunks:
 11


**Generate Chunk Embeddings**

In [73]:
chunk_embeddings = embedding_model.encode(chunks)
# chunk_embeddings

**Store Chunks in FAISS**

In [74]:
dimension = columns = chunk_embeddings.shape[1]
pdf_index = faiss.IndexFlatL2(dimension)
pdf_index.add(np.array(chunk_embeddings, dtype='float32'))
print("Total No. of Vectors Stored inside the PDF Data Index: ", pdf_index.ntotal)    #kati wota vector embeddings store vako

Total No. of Vectors Stored inside the PDF Data Index:  11


**Ask Questions on PDF**

In [75]:
query = "What is the document about?"
query_embedding = embedding_model.encode([query])

**Retrieve Relevant PDF Chunks**

In [76]:
distances, indices = pdf_index.search(np.array(query_embedding, dtype="float32"), 3)
print(distances)
print(indices[0])

[[1.6722624 1.7680931 1.7721221]]
[5 9 8]


In [77]:
retrieved_chunks = []
for i in indices[0]:
  # print(chunks[i])
  retrieved_chunks.append(chunks[i])
print(retrieved_chunks)
# Using List Comprehension:
# retrieved_chunks = [chunks[i] for i in indices[0]]

['sh-based: Based on source/destination IP or port. \nb. Load Balancers \n\uf0b7 Types: \no Hardware-based: Dedicated appliances (e.g., F5). \nhttps://www.f5.com/products/big-ip-services/iseries-appliance \no Software-based: Open-source or virtual (e.g., HAProxy, Nginx). \nhttps://www.openlogic.com/blog/nginx-vs', 's. \nc. Protocols Supporting High Availability \n\uf0b7 HSRP (Hot Standby Router Protocol) \n\uf0b7 VRRP (Virtual Router Redundancy Protocol) \n\uf0b7 GLBP (Gateway Load Balancing Protocol) \nExample Scenario: \nTwo routers configured with HSRP; if primary fails, secondary takes over IP without affecting \nusers. \n \nTuto', 'hieved by: \no Redundant hardware \no Failover systems \no Dynamic routing protocols (e.g., OSPF, EIGRP) \nb. Redundancy \n\uf0b7 Types: \no Link Redundancy: Multiple paths between nodes (e.g., dual links). o Device Redundancy: Backup routers/switches (e.g., HSRP, VRRP). \no Power Redundancy: UPS, redundant PSU']


**Generate Final RAG Answer**

In [78]:
context = "\n".join(retrieved_chunks)
messages = [
    {"role": "system", "content": "Answer only from the provided context"},
    {"role": "user", "content": f"Context: {context}\nQuestion: {query}"},
    ]

In [79]:
response = client.chat.completions.create(model = MODEL, messages = messages)
print(response)
print(response.choices)
print(response.choices[0])
print(response.choices[0].message)
print(response.choices[0].message.content)

ChatCompletion(id='chatcmpl-0dd02bfe-be6a-4ccb-9032-f1d1834e0250', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='The document is a concise overview of **high‑availability and redundancy in networking**. It covers:\n\n* Load‑balancing solutions (hardware‑based appliances such as F5 Big‑IP and software/virtual options like HAProxy and Nginx)  \n* Protocols that provide router/ gateway failover (HSRP, VRRP, GLBP) with an example of HSRP failover between two routers  \n* Redundancy strategies—including link redundancy, device redundancy, power redundancy, and the role of dynamic routing protocols (OSPF, EIGRP) and fail‑over systems.  \n\nIn short, it explains how to design networks that stay up and continue to serve traffic even when components fail.', role='assistant', annotations=None, executed_tools=None, function_call=None, reasoning='We need to answer based only on provided context. The question: "What is the document about?" The 